In [0]:
!pip install openpyxl

In [0]:
%restart_python

In [0]:
import pandas as pd

# Leer con pandas
pdf = pd.read_excel(
    "/Volumes/workspace/default/phisical_measures/SWaT_Dataset_Attack_v0.xlsx",
    header=1
)

print(f"Filas: {len(pdf):,}")
print(f"Columnas: {len(pdf.columns)}")
print(pdf.dtypes)

In [0]:
# Limpiar nombres de columnas — quitar espacios y caracteres inválidos para Delta
pdf.columns = [
    c.strip()                    # quitar espacios al inicio y al final
     .replace(" ", "_")          # espacios internos → guión bajo
     .replace("/", "_")          # slash → guión bajo
     .replace(";", "")
     .replace("{", "").replace("}", "")
     .replace("(", "").replace(")", "")
     .replace("\n", "").replace("\t", "")
     .replace("=", "")
    for c in pdf.columns
]

print("Columnas limpias:")
print(list(pdf.columns))

In [0]:
# Convertir a Spark si es necesario
df_attack = spark.createDataFrame(pdf)
display(df_attack)

In [0]:
# Convertir pandas a Spark y guardar como Delta
DELTA_PHYSICAL_PATH = "/Volumes/workspace/default/phisical_measures/delta/"


df_attack.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_PHYSICAL_PATH)

print(f"✅ Guardado en Delta Lake: {DELTA_PHYSICAL_PATH}")
print(f"Total filas: {df_attack.count():,}")
display(df_attack.limit(5))

In [0]:
pdf_normal = pd.read_excel(
    "/Volumes/workspace/default/phisical_measures/SWaT_Dataset_Normal_v0.xlsx",
    header=1
)

print(f"Filas: {len(pdf_normal):,}")
print(f"Columnas: {len(pdf_normal.columns)}")
print(pdf.dtypes)

# Limpiar nombres de columnas — quitar espacios y caracteres inválidos para Delta
pdf_normal.columns = [
    c.strip()                    # quitar espacios al inicio y al final
     .replace(" ", "_")          # espacios internos → guión bajo
     .replace("/", "_")          # slash → guión bajo
     .replace(";", "")
     .replace("{", "").replace("}", "")
     .replace("(", "").replace(")", "")
     .replace("\n", "").replace("\t", "")
     .replace("=", "")
    for c in pdf.columns
]

print("Columnas limpias:")
print(list(pdf_normal.columns))

In [0]:
# Convertir a Spark si es necesario
df_normal = spark.createDataFrame(pdf_normal)
display(df_normal)

In [0]:
# Convertir pandas a Spark y guardar como Delta
DELTA_PHYSICAL_PATH = "/Volumes/workspace/default/phisical_measures/delta/"


df_normal.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_PHYSICAL_PATH)

print(f"✅ Guardado en Delta Lake: {DELTA_PHYSICAL_PATH}")
print(f"Total filas: {df_normal.count():,}")
display(df_normal.limit(5))

In [0]:
print(f"Tipo actual: {df_normal.schema['Timestamp'].dataType}")
df_normal.select("Timestamp").show(5, truncate=False)

print(f"Tipo actual: {df_attack.schema['Timestamp'].dataType}")
df_attack.select("Timestamp").show(5, truncate=False)

In [0]:
# Unir ambos DataFrames
df_all = df_normal.unionByName(df_attack)

# Guardar como Delta
DELTA_ALL_PATH = "/Volumes/workspace/default/phisical_measures/delta_all/"

df_all.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_ALL_PATH)

print(f"✅ Guardado combinado en Delta Lake: {DELTA_ALL_PATH}")
print(f"Total filas: {df_all.count():,}")

# Mostrar ordenado por Timestamp
display(df_all.orderBy("Timestamp"))

In [0]:
from pyspark.sql import functions as F

def add_timestamp(df):
       
     return df.withColumn(
            "timestamp_dt",
            F.coalesce(
                F.expr("try_to_timestamp(trim(Timestamp), 'dd/MM/yyyy hh:mm:ss a')"),
                F.expr("try_to_timestamp(trim(Timestamp), 'd/MM/yyyy hh:mm:ss a')"),
                F.expr("try_to_timestamp(trim(Timestamp), 'dd/MM/yyyy h:mm:ss a')"),
                F.expr("try_to_timestamp(trim(Timestamp), 'd/MM/yyyy h:mm:ss a')"),
            )
        )

df_tm_normal = add_timestamp(df_normal)
df_tm_attack = add_timestamp(df_attack)

display(df_tm_normal.limit(5))

df_all_tm = add_timestamp(df_all)
display(df_all_tm.limit(5))

In [0]:
# Guardar como Delta
DELTA_ALL_PATH = "/Volumes/workspace/default/phisical_measures/delta_all/"

df_all_tm.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_ALL_PATH)

print(f"✅ Guardado combinado en Delta Lake: {DELTA_ALL_PATH}")
print(f"Total filas: {df_all_tm.count():,}")

# Mostrar ordenado por Timestamp
display(df_all_tm.orderBy("Timestamp"))


DELTA_PHYSICAL_PATH = "/Volumes/workspace/default/phisical_measures/delta_normal/"


df_tm_normal.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_PHYSICAL_PATH)

print(f"✅ Guardado en Delta Lake: {DELTA_PHYSICAL_PATH}")
print(f"Total filas: {df_tm_normal.count():,}")
display(df_tm_normal.limit(5))


DELTA_PHYSICAL_PATH = "/Volumes/workspace/default/phisical_measures/delta_attack/"


df_tm_attack.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_PHYSICAL_PATH)

print(f"✅ Guardado en Delta Lake: {DELTA_PHYSICAL_PATH}")
print(f"Total filas: {df_tm_attack.count():,}")
display(df_tm_attack.limit(5))
